# Успешность четырёх патчей на COCO_people

Парное сравнение на **всех валидных clean-изображениях**: изображение валидно, если YOLO11s на исходном изображении находит класс `person` при `conf=0.01`. Как и в текущей постановке проекта, патч вставляется в `(0, 0)` **после letterbox до 640×640**, а успех определяется как

$$\max conf_{person}(clean)-\max conf_{person}(patched) \ge 0.30.$$

Запускаются две парные постановки: патчи в нативном размере и все патчи после ресайза до 160×160. Первая измеряет практический эффект готовых файлов, вторая контролирует площадь и лучше сравнивает содержимое.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'CandidateRoutingAndAttackPath').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError('Не найден корень PatchSuccessResearch')

REPO_ROOT = find_repo_root()
VENDOR_DIR = REPO_ROOT / '.vendor'
if VENDOR_DIR.is_dir() and str(VENDOR_DIR) not in sys.path:
    sys.path.insert(0, str(VENDOR_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT =', REPO_ROOT)

## Конфигурация

Если COCO_people находится не в стандартном месте, измените только `DATASET_DIR`. `MAX_IMAGES=None` означает всю выборку; для быстрой проверки можно временно поставить, например, `64`.

In [ ]:
from CandidateRoutingAndAttackPath.multi_patch_asr import MultiPatchASRConfig

DATASET_CANDIDATES = [
    REPO_ROOT / 'datasets' / 'COCO_people',
    REPO_ROOT.parent / 'datasets' / 'COCO_people',
    REPO_ROOT.parent / 'COCO_people_640x640_cropped',
]
DATASET_DIR = next((p for p in DATASET_CANDIDATES if p.is_dir()), DATASET_CANDIDATES[0])
PATCH_PATHS = [
    REPO_ROOT / 'data' / '0709_yolo_dpatch_1000.png',
    REPO_ROOT / 'data' / 'cls_patch.png',
    REPO_ROOT / 'data' / 'depatch.png',
    REPO_ROOT / 'data' / 'nap1500new.png',
]
import torch
DEVICE = 'cuda:0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
MAX_IMAGES = None
FORCE = False

COMMON = dict(
    dataset_dir=str(DATASET_DIR), patch_paths=tuple(map(str, PATCH_PATHS)),
    model_path=str(REPO_ROOT / 'yolo11s.pt'),
    output_root=str(REPO_ROOT / 'CandidateRoutingAndAttackPath' / 'multi_patch_asr_outputs'),
    imgsz=640, conf=0.01, success_drop=0.30, patch_xy=(0, 0),
    batch_size=64 if DEVICE.startswith('cuda') else 32, device=DEVICE, max_images=MAX_IMAGES,
)
configs = {
    'native': MultiPatchASRConfig(**COMMON, patch_size=None),
    'resized_160': MultiPatchASRConfig(**COMMON, patch_size=(160, 160)),
}

print('dataset:', DATASET_DIR, 'exists =', DATASET_DIR.is_dir())
print('device:', DEVICE)
for path in PATCH_PATHS:
    from PIL import Image
    print(f'{path.stem:26s}', Image.open(path).size)
configs

## Визуальная проверка патчей

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(PATCH_PATHS), figsize=(14, 4), constrained_layout=True)
for axis, path in zip(axes, PATCH_PATHS):
    image = Image.open(path).convert('RGB')
    axis.imshow(image)
    axis.set_title(f'{path.stem}\n{image.width}×{image.height}px')
    axis.axis('off')
plt.show()

## Полный прогон

Clean baseline считается один раз и переиспользуется обеими постановками. Затем те же валидные изображения проходят с каждым патчем. Результаты кэшируются по хешам датасет-манифеста, модели, патчей и конфигурации. Повторный запуск с `FORCE=False` загружает CSV без инференса.

In [ ]:
from CandidateRoutingAndAttackPath.multi_patch_asr import evaluate_multi_patch_asr

import pandas as pd

runs = {}
detail_frames, summary_frames = [], []
for setting, setting_config in configs.items():
    print(f'\n=== {setting} ===')
    run = evaluate_multi_patch_asr(setting_config, force=FORCE)
    runs[setting] = run
    detail_frames.append(run['details'].assign(setting=setting))
    summary_frames.append(run['summary'].assign(setting=setting))
    print('loaded_from_cache =', run['loaded_from_cache'])
    print('clean_loaded_from_cache =', run['metadata'].get('clean_loaded_from_cache'))
    print('output_dir =', run['output_dir'])

details = pd.concat(detail_frames, ignore_index=True)
summary = pd.concat(summary_frames, ignore_index=True)
print('dataset images =', runs['native']['metadata']['dataset_images_total'])
print('valid clean images =', runs['native']['metadata']['clean_images_valid'])
summary.sort_values(['patch', 'setting']).style.format({
    'asr': '{:.3%}', 'asr_ci_low': '{:.3%}', 'asr_ci_high': '{:.3%}',
    'complete_hide_rate': '{:.3%}', 'mean_conf_clean': '{:.4f}',
    'mean_conf_patch': '{:.4f}', 'mean_drop': '{:.4f}',
    'median_drop': '{:.4f}', 'patch_area_frac': '{:.3%}',
})

## Прямое сравнение двух постановок

Этот график — главный контроль: изменение `native → resized_160` показывает вклад площади/масштаба для каждого содержимого патча.

In [ ]:
import numpy as np

patch_order = summary[summary['setting'].eq('resized_160')].sort_values('asr', ascending=False)['patch'].tolist()
fig, axis = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(patch_order))
width = 0.36
for offset, setting in zip([-width/2, width/2], ['native', 'resized_160']):
    current = summary[summary['setting'].eq(setting)].set_index('patch').loc[patch_order]
    yerr = np.vstack([current['asr'] - current['asr_ci_low'], current['asr_ci_high'] - current['asr']])
    axis.bar(x + offset, current['asr'], width, yerr=yerr, capsize=3, label=setting)
axis.set(xticks=x, xticklabels=patch_order, ylabel='ASR', ylim=(0, 1), title='Native size vs controlled 160×160')
axis.grid(axis='y', alpha=.25)
axis.legend()
plt.show()

comparison_path = Path(configs['native'].output_root) / 'native_vs_resized_160.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
print('saved:', comparison_path)

## Основные графики отдельно для каждой постановки

ASR показывается с 95% Wilson CI. Кроме среднего результата смотрим форму confidence drop и зависимость от размера исходного bbox человека.

In [ ]:
from CandidateRoutingAndAttackPath.multi_patch_asr import plot_multi_patch_results

for setting, run in runs.items():
    print(f'=== {setting} ===')
    fig = plot_multi_patch_results(run['details'], run['summary'], success_drop=configs[setting].success_drop)
    figure_path = Path(run['output_dir']) / 'multi_patch_asr_overview.png'
    fig.savefig(figure_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', figure_path)

## Насколько патчи ломают одни и те же изображения

Высокий overlap означает общий механизм/общую уязвимую подвыборку; низкий overlap при близком ASR означает разные режимы отказа. Это полезнее одного pooled ASR для следующего multi-patch student.

In [ ]:
from CandidateRoutingAndAttackPath.multi_patch_asr import plot_success_overlap

for setting, run in runs.items():
    print(f'=== {setting} ===')
    fig = plot_success_overlap(run['details'])
    overlap_path = Path(run['output_dir']) / 'success_overlap.png'
    fig.savefig(overlap_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', overlap_path)

## Диагностика по confidence clean

Этот график отделяет эффект на изначально слабых детекциях от эффекта на уверенных clean-примерах.

In [ ]:
diagnostic = details.copy()
diagnostic['clean_conf_bin'] = pd.cut(
    diagnostic['conf_clean'], bins=[0, .25, .50, .75, .90, 1.0], include_lowest=True
)
confidence_table = (
    diagnostic.groupby(['setting', 'clean_conf_bin', 'patch'], observed=True)
    .agg(n=('success', 'size'), asr=('success', 'mean'), mean_drop=('drop', 'mean'))
    .reset_index()
)
display(confidence_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True, constrained_layout=True)
for axis, setting in zip(axes, ['native', 'resized_160']):
    current = confidence_table[confidence_table['setting'].eq(setting)]
    for patch, group in current.groupby('patch'):
        axis.plot(group['clean_conf_bin'].astype(str), group['asr'], marker='o', label=patch)
    axis.set(title=setting, xlabel='clean max-person confidence', ylabel='ASR', ylim=(0, 1))
    axis.grid(alpha=.25)
    axis.tick_params(axis='x', rotation=20)
    axis.legend()
plt.show()

## Файлы результата

Каждая постановка получает отдельную папку результата.

- `details.csv`: одна строка на пару изображение × патч;
- `summary.csv`: итоговые ASR и интервалы;
- `clean_baseline.csv`: валидность и clean bbox;
- `metadata.json`: точная постановка;
- `failures.json`: ошибки чтения/инференса, если были;
- два PNG с основными графиками.